# HT

Extracting phenotype from GP, HES, and self-reported conditions (field 20002)

In [ ]:
import pandas as pd
import numpy as np
import os
import sys
pd.set_option('display.max_rows', 500)

In [ ]:


root_path = os.path.dirname(os.path.abspath(os.path.dirname('__file__')))
sys.path.insert(0, root_path)

In [ ]:
from env.parameters import P


In [ ]:
from phenotyping.codebase_phenotyping import (
pheno_all_evdt_extractor,
pheno_rank_and_filter,
pheno_keep_one_row,
clean_biobank_ado_pheno,
join_gp_hes_biobank_single_row_dfs,
join_single_row_dfs
)
from phenotyping.self_reported_phenotyping import (
self_reported_pheno_extractor,
qc_self_reported_against_cohort,
join_multiple_self_reported_phenos
)
from util.parquet_maker import dask_to_parquet
from util.general_utils import field_availability_check, list_field_instances
from util.dask_utils import import_field_from_dask
from util.parquet_maker import dask_to_parquet

In [ ]:
import dask.dataframe as dd

# Load main files

In [ ]:
# cohort
df_cohort = pd.read_csv(f'''{P.output_csv_path}/cohort_2_advanced.csv''')
df_cohort.head()

In [ ]:
# Dask-based GP data
dd_gp = dd.read_parquet(f'''{P.output_parquet_path}/gp_clinical_curated''')
dd_gp.head()

In [ ]:
# Select columns
df_gp = dd_gp[["eid", "read_curated", "evdt_gp"]].compute()
df_gp.dtypes

In [ ]:
# Read HES data
df_hes = pd.read_csv(f'''{P.output_csv_path}/hesin_diag_curated.csv''')
df_hes

# HT



In [ ]:
# Codelist
codelist_in = pd.read_csv(f'''{P.codelist_path}/curated/ht_updated.csv''')
codelist_in.head()

In [ ]:
# Name of the phenotype
assign_pheno_name = "ht"

In [ ]:
codelist_in["vocab"].value_counts(dropna=False)

In [ ]:
codelist_in["pheno_name"].value_counts(dropna=False)

## Extract HT in GP

In [ ]:
out_gp_long = pheno_all_evdt_extractor(df_data_clean=df_gp,
                                             col_evdt_data="evdt_gp",
                                             col_code_data="read_curated",
                                             df_codelist=codelist_in,
                                             col_code_codelist="code_clean",
                                             col_vocab_codelist="vocab",
                                             use_vocab="Read2",
                                             col_assign_pheno_name="pheno",
                                             assign_pheno_name=assign_pheno_name,
                                             col_codelist_multicategory=None,
                                             join_type="read2")

In [ ]:
out_gp_long.head()

In [ ]:
out_gp_long = out_gp_long[['eid', 'evdt_gp', 'pheno']]
out_gp_long.head(5)

In [ ]:
out_gp_long_ranked = pheno_rank_and_filter(out_gp_long, col_evdt="evdt_gp", col_eid="eid", earliest_ranks_1=True)
out_gp_long_ranked.head(10)


In [ ]:
# Use this later for joining with HES (and ADO, if any)
out_gp_single_row = pheno_keep_one_row(df_ranked_long=out_gp_long_ranked,
                                       col_row_number="row_number")
out_gp_single_row.head(10)

In [ ]:
 out_gp_single_row['eid'].count() == out_gp_single_row['eid'].nunique()

## HT in HES

In [ ]:
df_hes_long =pheno_all_evdt_extractor(df_data_clean=df_hes,
                                             col_evdt_data="epistart",
                                             col_code_data="diag_icd10",
                                             df_codelist=codelist_in,
                                             col_code_codelist="code_clean",
                                             col_vocab_codelist="vocab",
                                             use_vocab="ICD10",
                                             col_assign_pheno_name="pheno",
                                             assign_pheno_name=assign_pheno_name,
                                             col_codelist_multicategory=None,
                                             join_type="icd10")
df_hes_long.head()

In [ ]:
df_hes_long = df_hes_long[['eid', 'epistart', 'pheno']]
df_hes_long.head(5)

In [ ]:
out_hes_long_ranked = pheno_rank_and_filter(df_hes_long, col_evdt="epistart", col_eid="eid", earliest_ranks_1=True)
out_hes_long_ranked.head(10)

In [ ]:
# Use this later for joining with GP (and ADO, if any)
out_hes_single_row = pheno_keep_one_row(df_ranked_long=out_hes_long_ranked,
                                       col_row_number="row_number")
out_hes_single_row.head(10)

## Join  GP and HES phenotypes

In [ ]:
df_gp_hes = join_gp_hes_biobank_single_row_dfs(pheno_gp=out_gp_single_row, pheno_hes=out_hes_single_row,
                                   col_evdt_gp="evdt_gp", col_evdt_hes="epistart", pheno_name=assign_pheno_name, gp_and_hes_only=True, keep_minimum=True,col_eid="eid")

In [ ]:
df_gp_hes.head()

# Make a sinlge ouput file

In [ ]:
df_gp_hes = df_gp_hes[['eid', f'''evdt_{assign_pheno_name}''']]
df_gp_hes.head()

In [ ]:
# Save
df_gp_hes.to_csv(f'''{P.output_phenotypes_csv_path}/incident_{assign_pheno_name}_gp_hes.csv''', index=False)


In [ ]:
# Test 
df_gp_hes = pd.read_csv(f'''{P.output_phenotypes_csv_path}/incident_{assign_pheno_name}_gp_hes.csv''')
df_gp_hes.head()

In [ ]:
df_gp_hes.dtypes


# Self reported

From field 2002

In [ ]:
# All fields
dd_all = dd.read_parquet(f'''{P.output_parquet_path}/{P.core_all_name}''')
dd_all.head()

In [ ]:
cols_all = dd_all.columns

In [ ]:
# Extract field no 20002, self-reported non-cancer illness
search_term = "20002"
list_term_self_reported = list_field_instances(col_list=cols_all, search_for_field=search_term)
dd_reported = import_field_from_dask(dd_in= dd_all, field_name=search_term, eid_col="eid")
# to pandas
df_reported = dd_reported.compute()
df_reported["eid"] = df_reported["eid"].astype('int64')
for item in list_term_self_reported:
    df_reported[item] = df_reported[item].astype('Int64')

In [ ]:
# Date of the non-cancer illness
search_term = "20008"
list_term_self_reported_date = list_field_instances(col_list=cols_all, search_for_field=search_term)
dd_reported_date = import_field_from_dask(dd_in= dd_all, field_name=search_term, eid_col="eid")
# to pandas
df_reported_date = dd_reported_date.compute()
df_reported_date["eid"] = df_reported_date["eid"].astype('int64')
for item in list_term_self_reported_date:
    df_reported_date[item] = df_reported_date[item].astype('float')


In [ ]:
reported_coding = pd.read_csv(P.field_encoding_dict.get("non_cancer_self_reported").get("coding_file_path"), delimiter="\t")
reported_coding.head()

In [ ]:
# hypertension, essential hypertension tags
list_codes_self_reported = [1065, 1072]

dict_codes_reported = {}
for item in list_codes_self_reported:
    dict_codes_reported[item] = self_reported_pheno_extractor(df_reported, df_reported_date, code= item)[["eid", f'''evdt_{item}''']]

In [ ]:
for key in dict_codes_reported.keys():
    print(dict_codes_reported.get(key).head())

In [ ]:
for key in dict_codes_reported.keys():
    print(key)
    _ = qc_self_reported_against_cohort(dict_codes_reported.get(key), evdt_col=f'''evdt_{key}''', df_cohort=df_cohort, drop_wrong_dates=False)

In [ ]:
df_all_self_reported = join_multiple_self_reported_phenos(dict_codes_self_reported=dict_codes_reported)
df_all_self_reported.head()

In [ ]:
df_gp_hes.head()


In [ ]:
df_gp_hes = df_gp_hes.rename(columns={f'''evdt_{assign_pheno_name}''': f'''evdt_gp_hes_{assign_pheno_name}'''})
df_gp_hes.head()

In [ ]:
df_gp_hes[f'''evdt_gp_hes_{assign_pheno_name}'''] = pd.to_datetime(df_gp_hes[f'''evdt_gp_hes_{assign_pheno_name}'''])

In [ ]:
df_all = join_single_row_dfs(pheno_1= df_gp_hes,
                               pheno_2= df_all_self_reported,
                               col_evdt_1= f'''evdt_gp_hes_{assign_pheno_name}''',
                               col_evdt_2= "evdt_self_reported",
                               pheno_name= assign_pheno_name,
                               keep_extra_cols_1=[],
                               keep_extra_cols_2=[],
                               keep_minimum=True

)

In [ ]:
df_all.head()


In [ ]:
df_all[df_all["evdt_self_reported"].notnull()].head()


In [ ]:
df_out = df_all[["eid", f'''evdt_{assign_pheno_name}''']]
df_out.head()

In [ ]:
# Mixture of GP, HES, and self-reported non-cancer illness phenotypes
df_out.to_csv(f'''{P.output_phenotypes_csv_path}/incident_{assign_pheno_name}_gp_hes_self_reported.csv''', index=False)


In [ ]:
df_gp_hes.shape


In [ ]:
df_out.shape


In [ ]:
assign_pheno_name
